# Symptom Term Embedding Similarity

This notebook builds intuition for the semantic-mapping step described in the project README: extracted symptom expressions are mapped to SYMP ontology concepts using `intfloat/multilingual-e5-large` embeddings and a cosine-similarity threshold (0.90). It visualizes:

1. The angle between two embedding vectors (`back pain` vs. `backache`) and the cosine similarity it corresponds to,
2. A 2D PCA projection of several related and unrelated terms around a shared anchor, color-coded by similarity band.

## Package Installation

In [ ]:
%pip install sentence-transformers numpy matplotlib scikit-learn

## Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Arc, FancyArrowPatch, FancyBboxPatch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

In [ ]:
ANCHOR = "back pain"
TERMS = [
    "backache",
    "lumbar pain",
    "neck pain",
    "headache",
]
ALL_TERMS = [ANCHOR] + TERMS

BLUE = "#185FA5"
TEAL = "#0F6E56"
AMBER = "#BA7517"
CORAL = "#993C1D"
DARK = "#111827"
MUTED = "#6B7280"
LINE = "#E5E7EB"
PANEL = "#F8FAFC"

RESULTS_DIR = Path("results") / "embedding-similarity"
OUTPUT_FILENAME = "embedding_similarity.png"

## Loading the Embedding Model

Guarded by `try`/`except NameError` so re-running this cell during exploration doesn't reload the model into memory.

In [ ]:
try:
    model
except NameError:
    model = SentenceTransformer("intfloat/multilingual-e5-large")

## Encoding Terms and Computing Similarity

In [ ]:
embeddings = model.encode(ALL_TERMS, normalize_embeddings=True)
emb_anchor = embeddings[0]
cos_sims = [float(np.dot(emb_anchor, e)) for e in embeddings]


def angle_deg(sim):
    return np.degrees(np.arccos(np.clip(sim, -1.0, 1.0)))


def sim_to_color(sim):
    if sim > 0.92:
        return TEAL
    if sim > 0.85:
        return "#1D9E75"
    if sim > 0.78:
        return AMBER
    return CORAL

## Building and Saving the Comparison Figure

Two panels: (a) the angle between the anchor and a single closely related term, (b) a PCA projection of every term, colored by similarity band. Both panels share one `fig`/`ax1`/`ax2` state, so — unlike the rest of this notebook — this has to stay a single cell: Jupyter's inline backend auto-closes a figure once a cell that drew on it finishes running, which would silently blank out anything a later cell tried to add to the same `fig`.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2), dpi=220)
fig.patch.set_facecolor("white")


def card(ax, x, y, w, h):
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.04",
        linewidth=0.8, edgecolor=LINE, facecolor=PANEL, zorder=0,
    ))


# ── Panel A: vector angle ───────────────────────────────────────────────
ax1.set_facecolor("white")
card(ax1, -0.08, -0.12, 1.36, 0.82)

sim_b = cos_sims[1]  # backache
theta_b = np.arccos(np.clip(sim_b, -1.0, 1.0))
v_bx, v_by = np.cos(theta_b), np.sin(theta_b)

# Thin reference grid arcs
for r in [0.3, 0.6, 0.9]:
    ax1.add_patch(Arc((0, 0), 2 * r, 2 * r, angle=0, theta1=0, theta2=90,
                      linewidth=0.5, color=LINE, zorder=1))

# backache vector (teal)
ax1.add_patch(FancyArrowPatch(
    (0, 0), (v_bx * 0.90, v_by * 0.90),
    arrowstyle="-|>", mutation_scale=18,
    linewidth=2.8, color=TEAL, zorder=3,
))
ax1.text(v_bx * 0.95 + 0.03, v_by * 0.95 + 0.02, "backache",
         fontsize=10, fontweight="bold", color=TEAL,
         ha="left", va="bottom", zorder=4,
         bbox=dict(boxstyle="round,pad=0.22", facecolor="white",
                   edgecolor="#9FE1CB", linewidth=0.8))

# back pain vector (blue, horizontal)
ax1.add_patch(FancyArrowPatch(
    (0, 0), (0.92, 0),
    arrowstyle="-|>", mutation_scale=18,
    linewidth=2.8, color=BLUE, zorder=3,
))
ax1.text(0.95, -0.06, "back pain",
         fontsize=10, fontweight="bold", color=BLUE,
         ha="left", va="top", zorder=4,
         bbox=dict(boxstyle="round,pad=0.22", facecolor="white",
                   edgecolor="#B5D4F4", linewidth=0.8))

# Arc for theta
ax1.add_patch(Arc((0, 0), 0.34, 0.34, angle=0,
                  theta1=0, theta2=np.degrees(theta_b),
                  linewidth=2.0, color=AMBER, zorder=4))
# The theta glyph sits on the arc's angle bisector, centered (ha/va="center")
# so it doesn't default to baseline anchoring — which would draw it mostly
# above the point and, for an angle this acute, push it past the backache
# vector instead of resting cleanly in the wedge between the two vectors.
mid_a = theta_b / 2
label_r = 0.30
ax1.text(np.cos(mid_a) * label_r, np.sin(mid_a) * label_r,
         "θ", fontsize=13, fontweight="bold", color=AMBER,
         ha="center", va="center", zorder=5)

# cos sim + angle annotation
ax1.text(0.54, 0.38,
         f"cos(θ) = {sim_b:.3f}\nangle = {np.degrees(theta_b):.1f}°",
         fontsize=9.5, color=DARK, ha="center", va="center",
         linespacing=1.6,
         bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                   edgecolor=LINE, linewidth=0.9))

# Origin dot
ax1.scatter(0, 0, s=32, color=DARK, zorder=7)

ax1.set_title("(a) angle between embedding vectors", fontsize=10,
              fontweight="bold", color=DARK, loc="left", pad=8)
ax1.set_xlim(-0.10, 1.28)
ax1.set_ylim(-0.18, 0.80)
ax1.set_aspect("equal")
ax1.axis("off")


# ── Panel B: PCA projection ─────────────────────────────────────────────
ax2.set_facecolor("white")
card(ax2, -1.15, -0.60, 2.30, 1.20)

X_2d = PCA(n_components=2).fit_transform(embeddings)

# Draw connector lines from anchor to each point
for i in range(1, len(ALL_TERMS)):
    alpha = 0.15 + cos_sims[i] * 0.30
    ax2.plot(
        [X_2d[0, 0], X_2d[i, 0]],
        [X_2d[0, 1], X_2d[i, 1]],
        linestyle="--", linewidth=0.8,
        color=sim_to_color(cos_sims[i]), alpha=alpha, zorder=1,
    )

# Scatter all terms. Labels are pushed radially outward from the origin
# (PCA output is always mean-centered, so the origin is the cluster's
# centroid) and aligned away from their point, which keeps them from
# drifting toward each other — and toward the anchor — the way a fixed
# offset would whenever two terms land close together in the projection.
label_offset = 0.05
for i, (term, sim) in enumerate(zip(ALL_TERMS, cos_sims)):
    x, y = X_2d[i]
    color = BLUE if i == 0 else sim_to_color(sim)
    size = 160 if i == 0 else 80
    ax2.scatter(x, y, s=size, color=color,
                edgecolor="white", linewidth=1.2, zorder=3)

    dist_from_origin = np.hypot(x, y) or 1.0
    ux, uy = x / dist_from_origin, y / dist_from_origin
    label_x, label_y = x + ux * label_offset, y + uy * label_offset
    ha = "left" if ux >= 0 else "right"
    va = "bottom" if uy >= 0 else "top"
    sim_str = "" if i == 0 else f"  {sim:.3f}"

    ax2.text(
        label_x, label_y, term + sim_str,
        fontsize=8 if i > 0 else 9.5,
        fontweight="bold" if i == 0 else "normal",
        color=color, ha=ha, va=va, zorder=4,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                  edgecolor="none", alpha=0.82),
    )

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=BLUE,
           markersize=8, label="anchor"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=TEAL,
           markersize=7, label="near-synonyms (>0.92)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=AMBER,
           markersize=7, label="related terms (0.78-0.92)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CORAL,
           markersize=7, label="distant terms (<0.78)"),
]
ax2.legend(handles=legend_elements, loc="lower right", fontsize=7.5,
           framealpha=0.9, edgecolor=LINE, borderpad=0.7)

ax2.set_title("(b) PCA projection of all terms", fontsize=10,
              fontweight="bold", color=DARK, loc="left", pad=8)
ax2.set_xlabel("PC 1", fontsize=8.5, color=MUTED)
ax2.set_ylabel("PC 2", fontsize=8.5, color=MUTED)
ax2.grid(color=LINE, linewidth=0.7, alpha=0.8)
ax2.tick_params(axis="both", labelsize=7.5, colors=MUTED)
for spine in ax2.spines.values():
    spine.set_visible(False)

# Margin leaves room for the outward-pushed labels plus their text width,
# not just the bare data points.
margin = 0.40
ax2.set_xlim(X_2d[:, 0].min() - margin, X_2d[:, 0].max() + margin)
ax2.set_ylim(X_2d[:, 1].min() - margin, X_2d[:, 1].max() + margin)


# ── Save and display ────────────────────────────────────────────────────
fig.text(
    0.5, -0.015,
    "Cosine similarity is computed in the original high-dimensional space; "
    "panel (b) is a 2D PCA projection for visualization only.",
    ha="center", fontsize=8, color=MUTED,
)

plt.tight_layout(pad=1.4)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
output_path = RESULTS_DIR / OUTPUT_FILENAME
plt.savefig(output_path, dpi=220, bbox_inches="tight", facecolor="white")
print(f"Figure saved to '{output_path}'.")

plt.show()

## Similarity Summary

In [ ]:
print(f"\n{'─' * 48}")
print(f"{'Term':<18} {'cos sim':>8}  {'angle':>8}")
print(f"{'─' * 48}")
for term, sim in zip(ALL_TERMS, cos_sims):
    marker = " ← anchor" if term == ANCHOR else ""
    print(f"{term:<18} {sim:>8.4f}  {angle_deg(sim):>7.2f}°{marker}")
print(f"{'─' * 48}")